# Reproduce candidate 317 submission

Этот ноутбук воспроизводит выбранное решение `candidate 317`, которое имеет visible public score `90.78` и лучший известный local OOF MAPE среди подтвержденных public-90.78 кандидатов.

Требования воспроизводимости закрываются явно:

- показан список Python-библиотек и их версий;
- зафиксирован `RANDOM_SEED = 2026`;
- при последовательном выполнении ячеек формируется CSV, byte-to-byte идентичный выбранному файлу `selected_solution/submission_candidate_317.csv`.

In [1]:
from pathlib import Path
import os
import random
import sys
import platform
import subprocess
import json
import hashlib
import importlib.util
import importlib.metadata as metadata

import numpy as np
import pandas as pd

RANDOM_SEED = 2026
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

ROOT = Path.cwd().resolve()
while not (ROOT / "selected_solution").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Project root:", ROOT)
print("RANDOM_SEED:", RANDOM_SEED)

Python: 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 10:07:17) [Clang 14.0.6 ]
Platform: macOS-15.5-arm64-arm-64bit
Project root: /Users/levonkrtyan/Library/Mobile Documents/com~apple~CloudDocs/Growth Gradient
RANDOM_SEED: 2026


/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## Python environment

Ниже фиксируется окружение: сначала ключевые библиотеки решения, затем полный `pip freeze` для компьютера участника.

In [2]:
key_packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "joblib",
    "threadpoolctl",
    "nbformat",
    "notebook",
    "jupyter",
]

def package_version(name: str) -> str:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "not installed"

key_versions = pd.DataFrame(
    {"package": key_packages, "version": [package_version(pkg) for pkg in key_packages]}
)
display(key_versions)

freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True).splitlines()
freeze_df = pd.DataFrame({"pip_freeze": freeze})
display(freeze_df)

versions_path = ROOT / "selected_solution" / "reproduction_package_versions.csv"
freeze_df.to_csv(versions_path, index=False)
print("Saved full environment snapshot to", versions_path.relative_to(ROOT))

,package,version
0,numpy,1.26.4
1,pandas,3.0.2
2,scikit-learn,1.4.2
3,scipy,1.13.1
4,joblib,1.4.2
5,threadpoolctl,2.2.0
6,nbformat,5.9.2
7,notebook,7.3.2
8,jupyter,1.0.0


,pip_freeze
0,accelerate==1.13.0
1,ace_tools==0.0
2,addict==2.4.0
3,aext-assistant @ file:///private/var/folders/n...
4,aext-assistant-server @ file:///private/var/fo...
...,...
422,yarl==1.23.0
423,zict @ file:///Users/builder/cbouss/perseveran...
424,zipp @ file:///Users/builder/cbouss/perseveran...
425,zope.interface @ file:///Users/builder/cbouss/...


Saved full environment snapshot to selected_solution/reproduction_package_versions.csv


## Load reproducible inputs

Для воспроизводимости входные meta-таблицы сохранены в текущей ветке в `selected_solution/inputs/`. Это убирает зависимость от remote-ветки GitHub при проверке организаторами.

In [3]:
input_oof = ROOT / "selected_solution" / "inputs" / "oof_meta_inputs_v11_v14.csv.gz"
input_test = ROOT / "selected_solution" / "inputs" / "test_meta_inputs_v11_v14.csv.gz"
reference_submission = ROOT / "selected_solution" / "submission_candidate_317.csv"
manifest_path = ROOT / "selected_solution" / "manifest.json"
script_path = ROOT / "analysis" / "new_baseline" / "train_v15_meta_residual_on_265.py"

for path in [input_oof, input_test, reference_submission, manifest_path, script_path]:
    assert path.exists(), f"Missing required file: {path}"

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(json.dumps({
    "selected_candidate_id": manifest["selected_candidate_id"],
    "local_oof_mape": manifest["local_oof_mape"],
    "gamma": manifest["gamma"],
    "clip": manifest["clip"],
    "model_source": manifest["model_source"],
}, ensure_ascii=False, indent=2))

{
  "selected_candidate_id": 317,
  "local_oof_mape": 3.274858073509391,
  "gamma": 0.8,
  "clip": 0.008,
  "model_source": "huber030"
}


## Import solution code

Используем тот же код, что и в проекте: построение meta-ratio признаков, выбор feature columns, модель `HuberRegressor`, MAPE и evaluation logic.

In [4]:
spec = importlib.util.spec_from_file_location("v15_meta", script_path)
v15_meta = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(v15_meta)

v15_meta.set_random_seed(RANDOM_SEED)
print("Imported", script_path.relative_to(ROOT))
print("Script RANDOM_SEED:", v15_meta.RANDOM_SEED)

Imported analysis/new_baseline/train_v15_meta_residual_on_265.py
Script RANDOM_SEED: 2026


## Train the residual expert

Кандидат `317` соответствует source-модели `huber030`, затем применяется multiplicative correction к `pred265` с параметрами `gamma=0.80`, `clip=0.008`.

Мы обучаем только нужный expert `huber030`, а не весь grid search, потому что для воспроизведения выбранного CSV остальные кандидаты не нужны.

In [5]:
train = pd.read_csv(input_oof)
test = pd.read_csv(input_test)

train = v15_meta.add_meta_ratios(train)
test = v15_meta.add_meta_ratios(test)
numeric, cats = v15_meta.feature_columns(train)
x_cols = numeric + cats

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Features:", len(numeric), "numeric,", len(cats), "categorical")

y = train["actual_log_residual_vs_265"].to_numpy(float)
kind = manifest["model_source"]

oof_pred = np.zeros(len(train), dtype=float)
for fold in sorted(train["fold"].unique()):
    tr = train["fold"] != fold
    va = train["fold"] == fold
    model = v15_meta.make_model(kind, numeric, cats)
    model.fit(train.loc[tr, x_cols], y[tr])
    oof_pred[va] = model.predict(train.loc[va, x_cols])

final_model = v15_meta.make_model(kind, numeric, cats)
final_model.fit(train[x_cols], y)
test_pred_raw = final_model.predict(test[x_cols])

raw_oof = pd.DataFrame({"new_id": train["new_id"], "fold": train["fold"], kind: oof_pred})
raw_test = {kind: test_pred_raw}
summary, test_predictions = v15_meta.evaluate_candidates(train, test, raw_oof, raw_test)

target_name = f"{kind}_g{manifest['gamma']:.2f}_c{manifest['clip']:.3f}"
row = summary.loc[summary["name"].eq(target_name)].iloc[0]
display(row.to_frame("value"))

assert abs(float(row["oof_mape"]) - float(manifest["local_oof_mape"])) < 1e-9
assert abs(float(row["delta_vs_265_oof"]) - float(manifest["delta_vs_pred265_oof"])) < 1e-9
print("Local OOF metrics match manifest.")

Train shape: (18657, 63)
Test shape: (18657, 63)
Features: 44 numeric, 5 categorical


,value
name,huber030_g0.80_c0.008
source,huber030
gamma,0.8
clip,0.008
oof_mape,3.274858
delta_vs_265_oof,-0.039716
mean_abs_vs_265_pct,0.239898
p95_abs_vs_265_pct,0.73695
max_abs_vs_265_pct,0.803209
mean_delta_vs_265_pct,0.013546


Local OOF metrics match manifest.


## Generate submission CSV

Файл ниже является прогнозом для отправки: две колонки `new_id,rto`, 18 657 строк, без пропусков и отрицательных значений.

In [6]:
generated_submission = ROOT / "selected_solution" / "reproduced_submission_candidate_317.csv"

out = test[["new_id"]].copy()
out["rto"] = np.round(np.maximum(test_predictions[target_name], 0), 2)
out.to_csv(generated_submission, index=False)

print("Generated:", generated_submission.relative_to(ROOT))
print("Shape:", out.shape)
print("Columns:", list(out.columns))
print("NaN count:", int(out.isna().sum().sum()))
print("Negative rto count:", int((out["rto"] < 0).sum()))

assert out.shape == (18657, 2)
assert list(out.columns) == ["new_id", "rto"]
assert out.isna().sum().sum() == 0
assert (out["rto"] < 0).sum() == 0

Generated: selected_solution/reproduced_submission_candidate_317.csv
Shape: (18657, 2)
Columns: ['new_id', 'rto']
NaN count: 0
Negative rto count: 0


## Exact identity check

Финальная проверка: сгенерированный CSV должен byte-to-byte совпадать с выбранным файлом, который был зафиксирован как `candidate 317`.

In [7]:
def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

reference_hash = sha256(reference_submission)
generated_hash = sha256(generated_submission)

print("Reference:", reference_submission.relative_to(ROOT), reference_hash)
print("Generated :", generated_submission.relative_to(ROOT), generated_hash)

assert reference_submission.read_bytes() == generated_submission.read_bytes()
print("OK: generated submission is byte-to-byte identical to selected candidate 317.")

Reference: selected_solution/submission_candidate_317.csv b5902fe7c4c568f0b79e07054f492ee854d57b4f5a86ddea92c430bb0767e3a7
Generated : selected_solution/reproduced_submission_candidate_317.csv b5902fe7c4c568f0b79e07054f492ee854d57b4f5a86ddea92c430bb0767e3a7
OK: generated submission is byte-to-byte identical to selected candidate 317.
